In [ ]:
"""
6. 폰트 클러스터링 — Step 1~3 (스트리밍)
  1) is_telop=True의 CRAFT bbox 수집 (수집 중 1% 샘플링, 전체 RAM 적재 안 함)
  2+3) 프레임 그룹 단위로 크롭 → 이진화 → 임베딩 → 청크 저장
"""

import os
import json
import cv2
import random
import pickle
import numpy as np
import pandas as pd
import torch
from torchvision import transforms
import timm
from tqdm.auto import tqdm
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
FRAME_DIR = "./data/2_frame_files"
SAVE_DIR = "./data/6_font_clustering"

SAMPLE_RATIO = 0.01
IMG_SIZE = 224
BATCH_SIZE = 512
CHUNK_SIZE = 50000        # 5만 개마다 디스크에 flush
NUM_WORKERS = 32
IO_WORKERS = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "efficientnet_b0"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Config: {DEVICE}, backbone={BACKBONE}, sample={SAMPLE_RATIO*100:.0f}%")


# ──────────────────────────────────────────────
# Step 1: CRAFT bbox 수집 (수집 중 1% 샘플링)
# ──────────────────────────────────────────────
print("\n📋 Step 1: CRAFT bbox 수집 + 샘플링...")


def collect_channel(channel):
    """수집하면서 바로 1% 샘플링. 전체를 RAM에 올리지 않음."""
    rng = random.Random(hash(channel) + 42)

    ch_ocr = os.path.join(OCR_DIR, channel)
    ch_telop = os.path.join(TELOP_DIR, channel)
    if not os.path.isdir(ch_ocr) or not os.path.isdir(ch_telop):
        return [], 0

    results = []
    total = 0
    for fname in os.listdir(ch_telop):
        if not fname.endswith(".parquet"):
            continue
        video_name = fname[:-8]
        ocr_pq = os.path.join(ch_ocr, fname)
        telop_pq = os.path.join(ch_telop, fname)
        if not os.path.exists(ocr_pq):
            continue

        df_ocr = pd.read_parquet(ocr_pq)
        df_telop = pd.read_parquet(telop_pq)
        telop_map = {int(r["frame_num"]): r for _, r in df_telop.iterrows()}

        for _, ocr_row in df_ocr.iterrows():
            fnum = int(ocr_row["frame_num"])
            telop_row = telop_map.get(fnum)
            if telop_row is None:
                continue

            is_telop = json.loads(telop_row["is_telop"]) if isinstance(telop_row["is_telop"], str) else telop_row["is_telop"]
            telop_indices = set(idx for idx in range(len(is_telop)) if is_telop[idx])
            if not telop_indices:
                continue

            craft_texts = json.loads(ocr_row["craft_texts"]) if isinstance(ocr_row["craft_texts"], str) else ocr_row["craft_texts"]
            craft_xywha = json.loads(ocr_row["craft_xywha"]) if isinstance(ocr_row["craft_xywha"], str) else ocr_row["craft_xywha"]
            craft_parent_idx = json.loads(ocr_row["craft_parent_idx"]) if isinstance(ocr_row["craft_parent_idx"], str) else ocr_row["craft_parent_idx"]

            for ci in range(len(craft_xywha)):
                parent = craft_parent_idx[ci] if ci < len(craft_parent_idx) else -1
                if parent not in telop_indices:
                    continue
                total += 1
                if rng.random() >= SAMPLE_RATIO:
                    continue
                results.append({
                    "channel": channel,
                    "video_name": video_name,
                    "frame_num": fnum,
                    "craft_idx": ci,
                    "xywha": craft_xywha[ci],
                    "text": craft_texts[ci] if ci < len(craft_texts) else "",
                })
    return results, total


channels = sorted([d for d in os.listdir(TELOP_DIR) if os.path.isdir(os.path.join(TELOP_DIR, d))])

sampled = []
total_craft = 0
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(collect_channel, ch): ch for ch in channels}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="  채널 스캔"):
        results, count = fut.result()
        sampled.extend(results)
        total_craft += count

print(f"  전체 CRAFT bbox (telop): {total_craft:,}")
print(f"  샘플링: {len(sampled):,}개 (~{SAMPLE_RATIO*100:.0f}%)")

# 프레임별 그룹핑
frame_groups = defaultdict(list)
for i, s in enumerate(sampled):
    key = (s["channel"], s["video_name"], s["frame_num"])
    frame_groups[key].append((i, s))

print(f"  고유 프레임 수: {len(frame_groups):,}")


# ──────────────────────────────────────────────
# Step 2+3: 크롭 → 이진화 → 임베딩 (스트리밍)
# ──────────────────────────────────────────────
print(f"\n🔧 Step 2+3: 크롭 + 이진화 + 임베딩 ({DEVICE})...")

# 모델 로드
model = timm.create_model(BACKBONE, pretrained=True, num_classes=0)
model = model.to(DEVICE).eval()
embed_dim = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)).shape[1]
print(f"  모델: {BACKBONE}, embed_dim={embed_dim}")

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def crop_bbox(img, cx, cy, w, h, angle, pad_ratio=0.15):
    w_pad = int(w * (1 + pad_ratio))
    h_pad = int(h * (1 + pad_ratio))
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    M[0, 2] += w_pad / 2 - cx
    M[1, 2] += h_pad / 2 - cy
    crop = cv2.warpAffine(img, M, (w_pad, h_pad), flags=cv2.INTER_LINEAR)
    if crop.size == 0:
        return np.zeros((32, 32), dtype=np.uint8)
    return crop


def binarize(crop):
    if len(crop.shape) == 3:
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def process_frame_group(args):
    key, items = args
    channel, video_name, frame_num = key
    img_path = os.path.join(FRAME_DIR, channel, video_name, f"frame_{frame_num:08d}.jpg")

    if not os.path.exists(img_path):
        return [(idx, None) for idx, _ in items]

    img = cv2.imread(img_path)
    if img is None:
        return [(idx, None) for idx, _ in items]

    results = []
    for idx, s in items:
        cx, cy, w, h, angle = s["xywha"]
        crop = crop_bbox(img, cx, cy, w, h, angle)
        binary = binarize(crop)
        results.append((idx, binary))
    return results


def embed_batch(crops_batch):
    """이진화 크롭 리스트 → embedding numpy array"""
    tensors = []
    for binary in crops_batch:
        img_3ch = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)
        tensors.append(preprocess(img_3ch))
    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        feats = model(batch).cpu().numpy().astype(np.float16)
    return feats


# 스트리밍 처리: 크롭 → 버퍼 → 임베딩 → 청크 저장
tasks = list(frame_groups.items())

# 결과 버퍼
buf_indices = []    # sampled 내 인덱스
buf_crops = []      # 이진화 크롭
buf_embeddings = [] # 임베딩

chunk_id = 0
total_saved = 0
total_invalid = 0

# 메타 인덱스 매핑: sampled[i] → (chunk_id, chunk 내 위치)
index_map = {}


def flush_buffer():
    """버퍼를 임베딩 → 디스크 저장 → 클리어"""
    global chunk_id, total_saved, buf_indices, buf_crops, buf_embeddings

    if not buf_crops:
        return

    # 임베딩 추출
    all_embs = []
    for i in range(0, len(buf_crops), BATCH_SIZE):
        batch = buf_crops[i:i + BATCH_SIZE]
        embs = embed_batch(batch)
        all_embs.append(embs)
    all_embs = np.concatenate(all_embs, axis=0)

    # 저장
    chunk_path = os.path.join(SAVE_DIR, f"chunk_{chunk_id:04d}")
    np.save(f"{chunk_path}_emb.npy", all_embs)
    with open(f"{chunk_path}_crops.pkl", "wb") as f:
        pickle.dump(buf_crops, f)
    with open(f"{chunk_path}_meta.json", "w") as f:
        meta = [sampled[idx] for idx in buf_indices]
        json.dump(meta, f, ensure_ascii=False)

    # 인덱스 매핑 기록
    for pos, idx in enumerate(buf_indices):
        index_map[idx] = (chunk_id, pos)

    total_saved += len(buf_crops)
    chunk_id += 1

    # 클리어
    buf_indices.clear()
    buf_crops.clear()


with ThreadPoolExecutor(max_workers=IO_WORKERS) as pool:
    for results in tqdm(pool.map(process_frame_group, tasks),
                        total=len(tasks), desc="  처리"):
        for idx, binary in results:
            if binary is None:
                total_invalid += 1
                continue
            buf_indices.append(idx)
            buf_crops.append(binary)

            if len(buf_crops) >= CHUNK_SIZE:
                flush_buffer()

# 남은 버퍼 flush
flush_buffer()

# 인덱스 매핑 저장
with open(os.path.join(SAVE_DIR, "index_map.json"), "w") as f:
    json.dump({str(k): v for k, v in index_map.items()}, f)

print(f"\n  💾 저장 완료: {SAVE_DIR}")
print(f"     청크 수: {chunk_id}")
print(f"     유효 크롭: {total_saved:,}")
print(f"     무효 크롭: {total_invalid:,}")
print(f"     embed_dim: {embed_dim}")
print(f"\n✅ Step 1~3 완료. Step 4~5는 6_font_clustering_45.py로 실행하세요.")

✅ Config: cuda, backbone=efficientnet_b0, sample=1%

📋 Step 1: CRAFT bbox 목록 수집...


  채널 스캔:  22%|██▏       | 145/664 [03:00<19:34,  2.26s/it]  

In [ ]:
"""
6. 폰트 클러스터링 — Step 4~5: HDBSCAN + Gradio 뷰어
  청크 파일들을 로드하여 클러스터링 + 시각화
"""

import os
import json
import glob
import pickle
import cv2
import numpy as np
import gradio as gr
from PIL import Image, ImageDraw, ImageFont
from collections import defaultdict

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
SAVE_DIR = "./data/6_font_clustering"
NUM_WORKERS = 32

# ──────────────────────────────────────────────
# 청크 로드
# ──────────────────────────────────────────────
print("📂 청크 로드 중...")

emb_files = sorted(glob.glob(os.path.join(SAVE_DIR, "chunk_*_emb.npy")))
n_chunks = len(emb_files)
print(f"  청크 수: {n_chunks}")

# 임베딩: 전부 합치기 (클러스터링에 필요)
emb_list = []
for f in emb_files:
    emb_list.append(np.load(f))
embeddings = np.concatenate(emb_list, axis=0)
del emb_list
print(f"  embeddings: {embeddings.shape}")

# 메타: 전부 합치기
meta_all = []
for i in range(n_chunks):
    with open(os.path.join(SAVE_DIR, f"chunk_{i:04d}_meta.json")) as f:
        meta_all.extend(json.load(f))
print(f"  meta: {len(meta_all):,}개")

# 크롭: 청크별로 lazy load (RAM 절약)
# chunk_id → 파일 경로 + 오프셋 매핑
chunk_offsets = []  # (start_idx, end_idx, chunk_id)
offset = 0
for i in range(n_chunks):
    emb = np.load(os.path.join(SAVE_DIR, f"chunk_{i:04d}_emb.npy"))
    size = len(emb)
    chunk_offsets.append((offset, offset + size, i))
    offset += size

_crop_cache = {}  # chunk_id → list of crops


def load_crop(global_idx):
    """글로벌 인덱스로 크롭 이미지 로드 (청크별 캐시)"""
    for start, end, cid in chunk_offsets:
        if start <= global_idx < end:
            if cid not in _crop_cache:
                # 메모리 절약: 캐시는 최근 3개 청크만
                if len(_crop_cache) >= 3:
                    oldest = min(_crop_cache.keys())
                    del _crop_cache[oldest]
                with open(os.path.join(SAVE_DIR, f"chunk_{cid:04d}_crops.pkl"), "rb") as f:
                    _crop_cache[cid] = pickle.load(f)
            local_idx = global_idx - start
            return _crop_cache[cid][local_idx]
    return None


print(f"  총 샘플: {len(meta_all):,}")


# ──────────────────────────────────────────────
# Step 4: HDBSCAN 클러스터링
# ──────────────────────────────────────────────
print(f"\n📊 Step 4: HDBSCAN...")

from hdbscan import HDBSCAN

emb_f32 = embeddings.astype(np.float32)

clusterer = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    core_dist_n_jobs=NUM_WORKERS,
)
labels = clusterer.fit_predict(emb_f32)

n_clusters = labels.max() + 1
n_outliers = (labels == -1).sum()
print(f"  클러스터 수: {n_clusters}")
print(f"  outlier: {n_outliers:,}개 ({n_outliers / len(labels) * 100:.1f}%)")

for cid in range(n_clusters):
    count = (labels == cid).sum()
    print(f"  Cluster {cid:>3d}: {count:>6,}개")

np.save(os.path.join(SAVE_DIR, "cluster_labels.npy"), labels)
print(f"  💾 클러스터 라벨 저장 완료")


# ──────────────────────────────────────────────
# Step 5: Gradio 뷰어
# ──────────────────────────────────────────────
print("\n🎨 Step 5: Gradio 뷰어 시작...")

_FONT = None


def _get_font(size=14):
    global _FONT
    if _FONT is not None and _FONT.size == size:
        return _FONT
    for p in [
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJKkr-Regular.otf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    ]:
        if os.path.exists(p):
            _FONT = ImageFont.truetype(p, size)
            return _FONT
    _FONT = ImageFont.load_default()
    return _FONT


# 클러스터별 인덱스
cluster_indices = defaultdict(list)
for i, cid in enumerate(labels):
    cluster_indices[int(cid)].append(i)


def make_grid(cluster_id, page=0, per_page=30, cols=6):
    indices = cluster_indices.get(cluster_id, [])
    if not indices:
        return None, "빈 클러스터"

    start = page * per_page
    end = min(start + per_page, len(indices))
    page_indices = indices[start:end]

    if not page_indices:
        return None, "더 이상 샘플 없음"

    cell_h, cell_w = 80, 200
    rows_needed = (len(page_indices) + cols - 1) // cols
    grid = np.ones((rows_needed * (cell_h + 30), cols * (cell_w + 10), 3), dtype=np.uint8) * 255

    font = _get_font(12)
    pil_grid = Image.fromarray(grid)
    draw = ImageDraw.Draw(pil_grid)

    for gi, idx in enumerate(page_indices):
        row, col = gi // cols, gi % cols
        y_off = row * (cell_h + 30)
        x_off = col * (cell_w + 10)

        binary = load_crop(idx)
        if binary is None:
            continue

        h, w = binary.shape[:2]
        scale = min(cell_w / max(w, 1), cell_h / max(h, 1))
        new_w, new_h = int(w * scale), int(h * scale)
        if new_w > 0 and new_h > 0:
            resized = cv2.resize(binary, (new_w, new_h))
            rgb = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB)
            pad_x = (cell_w - new_w) // 2
            pad_y = (cell_h - new_h) // 2
            grid_np = np.array(pil_grid)
            grid_np[y_off + pad_y:y_off + pad_y + new_h, x_off + pad_x:x_off + pad_x + new_w] = rgb
            pil_grid = Image.fromarray(grid_np)
            draw = ImageDraw.Draw(pil_grid)

        meta = meta_all[idx]
        label = f"{meta['text'][:15]}"
        draw.text((x_off + 2, y_off + cell_h + 2), label, fill=(0, 0, 0), font=font)

    total_pages = (len(indices) + per_page - 1) // per_page
    info = (
        f"Cluster {cluster_id}: {len(indices):,}개 샘플\n"
        f"Page {page + 1}/{total_pages}\n"
        f"(전체: {n_clusters}개 클러스터, {n_outliers:,}개 outlier)"
    )

    return np.array(pil_grid), info


def on_view(cluster_id, page):
    img, info = make_grid(int(cluster_id), int(page))
    return img, info


with gr.Blocks(title="폰트 클러스터링 뷰어", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔤 폰트 클러스터링 뷰어 (HDBSCAN)")
    gr.Markdown(
        f"전체 {len(meta_all):,}개 샘플  |  "
        f"{n_clusters}개 클러스터  |  "
        f"{n_outliers:,}개 outlier (-1)"
    )

    with gr.Row():
        with gr.Column(scale=1):
            sl_cluster = gr.Slider(
                -1, n_clusters - 1, value=0, step=1,
                label="클러스터 ID (-1 = outlier)"
            )
            sl_page = gr.Slider(0, 1000, value=0, step=1, label="페이지")
            btn = gr.Button("🔍 보기", variant="primary")
            txt_info = gr.Textbox(label="정보", lines=4, interactive=False)

        with gr.Column(scale=3):
            img_out = gr.Image(label="클러스터 샘플", height=600)

    btn.click(on_view, [sl_cluster, sl_page], [img_out, txt_info])
    sl_cluster.change(on_view, [sl_cluster, sl_page], [img_out, txt_info])
    sl_page.change(on_view, [sl_cluster, sl_page], [img_out, txt_info])

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)

In [2]:
"""
CRAFT bbox 텍스트 분포 확인
- is_telop=True인 OCR bbox에 속한 CRAFT 글자별 빈도
"""

import os
import json
import pandas as pd
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
NUM_WORKERS = 32


def count_channel(channel):
    ch_ocr = os.path.join(OCR_DIR, channel)
    ch_telop = os.path.join(TELOP_DIR, channel)
    if not os.path.isdir(ch_ocr) or not os.path.isdir(ch_telop):
        return Counter()

    counter = Counter()
    for fname in os.listdir(ch_telop):
        if not fname.endswith(".parquet"):
            continue
        ocr_pq = os.path.join(ch_ocr, fname)
        telop_pq = os.path.join(ch_telop, fname)
        if not os.path.exists(ocr_pq):
            continue

        df_ocr = pd.read_parquet(ocr_pq)
        df_telop = pd.read_parquet(telop_pq)
        telop_map = {int(r["frame_num"]): r for _, r in df_telop.iterrows()}

        for _, ocr_row in df_ocr.iterrows():
            fnum = int(ocr_row["frame_num"])
            telop_row = telop_map.get(fnum)
            if telop_row is None:
                continue

            is_telop = json.loads(telop_row["is_telop"]) if isinstance(telop_row["is_telop"], str) else telop_row["is_telop"]
            telop_indices = set(idx for idx in range(len(is_telop)) if is_telop[idx])
            if not telop_indices:
                continue

            craft_texts = json.loads(ocr_row["craft_texts"]) if isinstance(ocr_row["craft_texts"], str) else ocr_row["craft_texts"]
            craft_parent_idx = json.loads(ocr_row["craft_parent_idx"]) if isinstance(ocr_row["craft_parent_idx"], str) else ocr_row["craft_parent_idx"]

            for ci in range(len(craft_texts)):
                parent = craft_parent_idx[ci] if ci < len(craft_parent_idx) else -1
                if parent not in telop_indices:
                    continue
                text = craft_texts[ci]
                if len(text) != 1:
                    continue
                counter[text] += 1
    return counter


channels = sorted([d for d in os.listdir(TELOP_DIR) if os.path.isdir(os.path.join(TELOP_DIR, d))])

total = Counter()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(count_channel, ch): ch for ch in channels}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="채널 스캔"):
        total += fut.result()

grand_total = sum(total.values())
print(f"\n전체 CRAFT bbox: {grand_total:,}")
print(f"고유 텍스트 수: {len(total):,}")

print(f"\n{'순위':>4}  {'텍스트':>6}  {'개수':>12}  {'비중':>7}  {'누적':>7}")
print("─" * 50)
cumul = 0
for rank, (text, count) in enumerate(total.most_common(), 1):
    pct = count / grand_total * 100
    cumul += pct
    print(f"{rank:>4}  {text:>6}  {count:>12,}  {pct:>6.2f}%  {cumul:>6.1f}%")

채널 스캔: 100%|██████████| 664/664 [01:04<00:00, 10.36it/s]



전체 CRAFT bbox: 670,069,771
고유 텍스트 수: 1,869

  순위     텍스트            개수       비중       누적
──────────────────────────────────────────────────
   1       이    16,225,858    2.42%     2.4%
   2       는     8,823,759    1.32%     3.7%
   3       e     7,537,714    1.12%     4.9%
   4       가     6,741,830    1.01%     5.9%
   5       다     6,685,931    1.00%     6.9%
   6       아     6,414,286    0.96%     7.8%
   7       지     5,995,646    0.89%     8.7%
   8       o     5,964,224    0.89%     9.6%
   9       고     5,894,792    0.88%    10.5%
  10       하     5,875,052    0.88%    11.4%
  11       에     5,743,488    0.86%    12.2%
  12       의     5,573,770    0.83%    13.1%
  13       어     5,355,526    0.80%    13.9%
  14       기     5,302,760    0.79%    14.6%
  15       T     5,285,159    0.79%    15.4%
  16       a     5,268,209    0.79%    16.2%
  17       0     5,234,433    0.78%    17.0%
  18       O     5,202,974    0.78%    17.8%
  19       S     5,156,742    0.77%    18.5%
  20